# Usage
- select the job_gpu_preemtable
- gpu: A100
- instance size: medium


# Import

# Camera–LiDAR fusion model on nuScenes

## Goal

This notebook documents the setup of a camera–LiDAR fusion 3D object detector on nuScenes using MMDetection3D.

The objective is to train a BEVFusion model that can be directly compared against the LiDAR-only baseline and used to test whether combining camera and LiDAR information improves detection performance.

## Experimental setting

The fusion model is trained on:
- the nuScenes dataset prepared in the previous notebook
- the same reproducible 20% training-scene subset used for the baseline
- the standard validation split

## Why this fusion model matters

A camera–LiDAR fusion model is important because:
- it tests whether complementary sensor information improves 3D detection
- it allows direct comparison with the LiDAR-only baseline
- it is motivated by the EDA, which suggests that small objects, pedestrian-related classes, and distant objects are likely to benefit from fusion
- BEVFusion provides a unified bird’s-eye-view representation that integrates LiDAR geometry and camera semantics

## Planned outcome

This notebook should:
1. verify the MMDetection3D environment for fusion experiments
2. identify the BEVFusion config to use
3. confirm the subset annotation file paths
4. create a fusion training config for the reduced dataset
5. document the training command
6. prepare the experiment for reproducible execution

In [1]:
# Core libraries
import shutil

from pathlib import Path
from typing import List, Tuple, Final

# OpenMMLab stack
import mmengine
import mmdet
import mmdet3d

# Setup and environment

## Define project paths

This notebook uses the same project and dataset paths as the dataset-preparation notebook.

They are defined explicitly here so that the notebook is self-contained and can be run independently.

In [2]:
# Project paths
PROJECT_ROOT: Final[Path] = Path("/storage/homefs/ae04q066/projects/nuscenes-multimodal-learning")
MMDET3D_ROOT: Final[Path] = PROJECT_ROOT / "external" / "mmdetection3d"

print("PROJECT_ROOT :", PROJECT_ROOT)
print("MMDET3D_ROOT :", MMDET3D_ROOT)

PROJECT_ROOT : /storage/homefs/ae04q066/projects/nuscenes-multimodal-learning
MMDET3D_ROOT : /storage/homefs/ae04q066/projects/nuscenes-multimodal-learning/external/mmdetection3d


## Verify software environment

Before configuring the fusion model, I verify that:
- MMDetection3D is accessible
- the correct Python environment is active
- core libraries are properly installed

This ensures that training will run without unexpected environment issues.

In [3]:
print("mmengine version:", mmengine.__version__)
print("mmdet version   :", mmdet.__version__)
print("mmdet3d version :", mmdet3d.__version__)

mmengine version: 0.10.7
mmdet version   : 3.2.0
mmdet3d version : 1.4.0


The MMDetection3D environment is correctly set up.

Library versions:
- mmengine: 0.10.7
- mmdet: 3.2.0
- mmdet3d: 1.4.0

These versions are compatible and confirm that the training pipeline can be executed.

## Select the model

For the fusion experiment, I use the **BEVFusion** detector, which combines LiDAR point clouds with multi-view camera images in a unified bird’s-eye-view representation for 3D object detection on nuScenes.

**Why BEVFusion**
- explicitly designed for camera–LiDAR fusion on nuScenes
- officially provided in the current MMDetection3D installation
- uses a BEV representation that naturally aligns LiDAR geometry and camera semantics
- enables a direct and fair comparison with the LiDAR-only baseline

**Configuration choice**

I use the official MMDetection3D BEVFusion config:

`bevfusion_lidar-cam_voxel0075_second_secfpn_8xb4-cyclic-20e_nus-3d.py`

This config defines:
- LiDAR-based feature extraction from point clouds
- multi-view camera feature extraction
- fusion in bird’s-eye-view (BEV) space
- a detection head operating on fused BEV features

# Experiment definition

The default training schedule runs for 20 epochs with periodic validation during training. In practice, this results in a runtime of up to 11 days on a single H100 GPU.

To make experimentation feasible, I use a reduced 20% subset of the dataset and adopt a lighter training schedule to:
- reduce wall-clock training time
- allow multiple experiments (baseline, fusion, ablations)
- still obtain meaningful results

This configuration is suitable for:
- debugging the pipeline
- obtaining a first BEVFusion result quickly
- enabling controlled comparisons with the LiDAR-only baseline

## Define the goal of this experiment

The goal of this experiment is to train a **BEVFusion camera–LiDAR model** on nuScenes and evaluate whether combining modalities improves detection performance compared to the LiDAR-only baseline, particularly in challenging scenarios such as small or distant objects.

In [5]:
EXPERIMENT_NAME: Final[str] = "bevfusion_nuscenes_20pct"

print("EXPERIMENT_NAME:", EXPERIMENT_NAME)

EXPERIMENT_NAME: bevfusion_nuscenes_20pct


## Select the model

I use **BEVFusion** as the fusion detector. It combines LiDAR point clouds with multi-view camera images in a unified bird’s-eye-view representation for 3D object detection on nuScenes.

**Rationale**

- explicitly designed for camera–LiDAR fusion on nuScenes
- supported in the current MMDetection3D installation
- enables direct comparison with the LiDAR-only baseline

In [6]:
MODEL_NAME: Final[str] = "BEVFusion"
MODEL_CONFIG_NAME: Final[str] = "bevfusion_lidar-cam_voxel0075_second_secfpn_8xb4-cyclic-20e_nus-3d.py"

print("MODEL_NAME       :", MODEL_NAME)
print("MODEL_CONFIG_NAME:", MODEL_CONFIG_NAME)

MODEL_NAME       : BEVFusion
MODEL_CONFIG_NAME: bevfusion_lidar-cam_voxel0075_second_secfpn_8xb4-cyclic-20e_nus-3d.py


## Define the configuration path

I create a separate experiment config instead of modifying the original MMDetection3D config directly. This keeps the repository clean and makes the experiment easier to reproduce.



In [7]:
SOURCE_CONFIG_PATH: Path = (
    MMDET3D_ROOT
    / "projects"
    / "BEVFusion"
    / "configs"
    / MODEL_CONFIG_NAME
)

EXPERIMENT_CONFIG_DIR: Path = MMDET3D_ROOT / "configs" / "my_experiments"
EXPERIMENT_CONFIG_DIR.mkdir(parents=True, exist_ok=True)

EXPERIMENT_CONFIG_PATH: Path = EXPERIMENT_CONFIG_DIR / "bevfusion_nuscenes_20pct.py"

print("SOURCE_CONFIG_PATH     :", SOURCE_CONFIG_PATH)
print("SOURCE exists          :", SOURCE_CONFIG_PATH.exists())
print("EXPERIMENT_CONFIG_DIR  :", EXPERIMENT_CONFIG_DIR)
print("EXPERIMENT_CONFIG_PATH :", EXPERIMENT_CONFIG_PATH)

SOURCE_CONFIG_PATH     : /storage/homefs/ae04q066/projects/nuscenes-multimodal-learning/external/mmdetection3d/projects/BEVFusion/configs/bevfusion_lidar-cam_voxel0075_second_secfpn_8xb4-cyclic-20e_nus-3d.py
SOURCE exists          : True
EXPERIMENT_CONFIG_DIR  : /storage/homefs/ae04q066/projects/nuscenes-multimodal-learning/external/mmdetection3d/configs/my_experiments
EXPERIMENT_CONFIG_PATH : /storage/homefs/ae04q066/projects/nuscenes-multimodal-learning/external/mmdetection3d/configs/my_experiments/bevfusion_nuscenes_20pct.py


## Copy the fusion config into the experiment folder

I now create a project-specific copy of the BEVFusion config.

This copied config will be the one modified for the reduced dataset experiment.
The original MMDetection3D config remains untouched.

In [8]:
if EXPERIMENT_CONFIG_PATH.exists():
    print("EXPERIMENT_CONFIG_PATH:", EXPERIMENT_CONFIG_PATH)
    print("\nSafety rule: existing file is not overwritten automatically.")
else:
    shutil.copy2(SOURCE_CONFIG_PATH, EXPERIMENT_CONFIG_PATH)
    print("Copied BEVFusion config to:")
    print(EXPERIMENT_CONFIG_PATH)

print("\nExperiment config exists:", EXPERIMENT_CONFIG_PATH.exists())

EXPERIMENT_CONFIG_PATH: /storage/homefs/ae04q066/projects/nuscenes-multimodal-learning/external/mmdetection3d/configs/my_experiments/bevfusion_nuscenes_20pct.py

Safety rule: existing file is not overwritten automatically.

Experiment config exists: True


# Config file settings

In this section, I prepare and modify the MMDetection3D config file for the fusion experiment.  
All key settings — dataset, modalities, training schedule, and data loading — are defined through this config.

The main changes define a reduced-budget training setup: a reproducible 20% training subset, a shorter training schedule, and adjusted data-loading settings. These changes reduce compute cost while keeping the experiment suitable for analysis.

The configuration is based on an official BEVFusion nuScenes setup and is adapted to match the reduced dataset and training constraints while remaining consistent with the LiDAR-only baseline.

## Open the config file

In [10]:
with open(EXPERIMENT_CONFIG_PATH, "r") as f:
    lines = f.readlines()

for i, line in enumerate(lines):
    print(f"{i+1:03d}: {line.rstrip()}")

001: _base_ = [
002:     '../../projects/BEVFusion/configs/bevfusion_lidar-cam_voxel0075_second_secfpn_8xb4-cyclic-20e_nus-3d.py'
003: ]
004: 
005: train_dataloader = dict(
006:     batch_size=2,
007:     num_workers=8,
008:     persistent_workers=True,
009:     dataset=dict(
010:         dataset=dict(
011:             ann_file='subsets/nuscenes_infos_train_20pct.pkl'
012:         )
013:     )
014: )
015: 
016: val_dataloader = dict(
017:     num_workers=8,
018:     persistent_workers=True
019: )
020: 
021: test_dataloader = dict(
022:     num_workers=8,
023:     persistent_workers=True
024: )
025: 
026: train_cfg = dict(by_epoch=True, max_epochs=5, val_interval=1)
027: 
028: param_scheduler = [
029:     dict(
030:         type='LinearLR',
031:         start_factor=0.33333333,
032:         by_epoch=False,
033:         begin=0,
034:         end=500
035:     ),
036:     dict(
037:         type='CosineAnnealingLR',
038:         begin=0,
039:         T_max=5,
040:         end=5,
041:      

## Define override values

I define the main settings that will override the source config for this experiment. These values describe the reduced-budget training setup used in this notebook.

They include adjustments to the training subset, training schedule, and data-loading parameters, ensuring consistency with the LiDAR-only baseline while preserving the official BEVFusion camera–LiDAR setup.

In [11]:
TRAIN_ANN_FILE: Final[str] = "subsets/nuscenes_infos_train_20pct.pkl"

MAX_EPOCHS: Final[int] = 5
VAL_INTERVAL: Final[int] = 1
TRAIN_CFG: Final[str] = f"dict(by_epoch=True, max_epochs={MAX_EPOCHS}, val_interval={VAL_INTERVAL})"

# BEVFusion is more memory-intensive than LiDAR-only models
TRAIN_BATCH_SIZE: Final[int] = 2
TRAIN_NUM_WORKERS: Final[int] = 8

print("Override values:")
print("TRAIN_ANN_FILE    :", TRAIN_ANN_FILE)
print("MAX_EPOCHS        :", MAX_EPOCHS)
print("VAL_INTERVAL      :", VAL_INTERVAL)
print("TRAIN_CFG         :", TRAIN_CFG)
print("TRAIN_BATCH_SIZE  :", TRAIN_BATCH_SIZE)
print("TRAIN_NUM_WORKERS :", TRAIN_NUM_WORKERS)

Override values:
TRAIN_ANN_FILE    : subsets/nuscenes_infos_train_20pct.pkl
MAX_EPOCHS        : 5
VAL_INTERVAL      : 1
TRAIN_CFG         : dict(by_epoch=True, max_epochs=5, val_interval=1)
TRAIN_BATCH_SIZE  : 2
TRAIN_NUM_WORKERS : 8


## Write a clear and self-contained experiment config

I define an explicit experiment config that includes all required dataset, modality, and pipeline components, while controlling the key experiment settings through a small set of variables. This includes the reduced training subset, the shorter training schedule, the data-loading parameters, and a checkpoint configuration that saves progress regularly during training.

The configuration is based on an official BEVFusion nuScenes setup and is adapted only where necessary to match the reduced dataset and training constraints.

**Rationale**

- keeps the configuration fully explicit and easy to understand  
- makes the experiment reproducible and self-contained  
- allows quick adjustments through a small set of variables  
- simplifies reuse of the same setup across different models  
- adds regular checkpoints for safer long HPC runs  

In [12]:
CONFIG_TEXT: str = f"""_base_ = [
    '../../projects/BEVFusion/configs/{MODEL_CONFIG_NAME}'
]

train_dataloader = dict(
    batch_size={TRAIN_BATCH_SIZE},
    num_workers={TRAIN_NUM_WORKERS},
    persistent_workers=True,
    dataset=dict(
        dataset=dict(
            ann_file='{TRAIN_ANN_FILE}'
        )
    )
)

val_dataloader = dict(
    num_workers={TRAIN_NUM_WORKERS},
    persistent_workers=True
)

test_dataloader = dict(
    num_workers={TRAIN_NUM_WORKERS},
    persistent_workers=True
)

train_cfg = {TRAIN_CFG}

param_scheduler = [
    dict(
        type='LinearLR',
        start_factor=0.33333333,
        by_epoch=False,
        begin=0,
        end=500
    ),
    dict(
        type='CosineAnnealingLR',
        begin=0,
        T_max={MAX_EPOCHS},
        end={MAX_EPOCHS},
        by_epoch=True,
        eta_min_ratio=1e-4,
        convert_to_iter_based=True
    ),
    dict(
        type='CosineAnnealingMomentum',
        eta_min=0.85 / 0.95,
        begin=0,
        end={MAX_EPOCHS * 0.4},
        by_epoch=True,
        convert_to_iter_based=True
    ),
    dict(
        type='CosineAnnealingMomentum',
        eta_min=1,
        begin={MAX_EPOCHS * 0.4},
        end={MAX_EPOCHS},
        by_epoch=True,
        convert_to_iter_based=True
    )
]

default_hooks = dict(
    checkpoint=dict(
        type='CheckpointHook',
        interval=1,
        save_last=True,
        max_keep_ckpts=3
    )
)
"""

In [13]:
with open(EXPERIMENT_CONFIG_PATH, "w") as f:
    f.write(CONFIG_TEXT)

print("Experiment config written to:")
print(EXPERIMENT_CONFIG_PATH)

Experiment config written to:
/storage/homefs/ae04q066/projects/nuscenes-multimodal-learning/external/mmdetection3d/configs/my_experiments/bevfusion_nuscenes_20pct.py


## Verify updated config settings

I verify that the main override settings were correctly written into the config file. Only the relevant lines are displayed to keep the output concise.

This step ensures that the reduced training subset, training schedule, and data-loading parameters have been correctly applied on top of the base BEVFusion configuration.

In [14]:
KEYWORDS = [
    "ann_file",
    "train_cfg",
    "batch_size",
    "num_workers",
    "T_max",          # uses MAX_EPOCHS
]

with open(EXPERIMENT_CONFIG_PATH, "r") as f:
    lines = f.readlines()

print("Updated config lines:\n")

for i, line in enumerate(lines):
    for keyword in KEYWORDS:
        if keyword in line:
            print(f"{i+1:03d}: {line.rstrip()}")
            # also show next line for context (important for nested dicts)
            if i + 1 < len(lines):
                print(f"{i+2:03d}: {lines[i+1].rstrip()}")
            print()
            break

Updated config lines:

006:     batch_size=2,
007:     num_workers=8,

007:     num_workers=8,
008:     persistent_workers=True,

011:             ann_file='subsets/nuscenes_infos_train_20pct.pkl'
012:         )

017:     num_workers=8,
018:     persistent_workers=True

022:     num_workers=8,
023:     persistent_workers=True

026: train_cfg = dict(by_epoch=True, max_epochs=5, val_interval=1)
027: 

039:         T_max=5,
040:         end=5,



# Training command

The BEVFusion model can be trained using the following command:

```bash
cd /storage/homefs/ae04q066/projects/nuscenes-multimodal-learning/external/mmdetection3d

conda activate py38_mmdet3d

export PYTHONPATH=$PWD:$PYTHONPATH

python tools/train.py configs/my_experiments/bevfusion_nuscenes_20pct.py

# Running the fusion training with SLURM

For long BEVFusion training runs on UBELIX, it is safer to submit the experiment through SLURM instead of launching it from a fragile interactive terminal.

**Why this is useful**

Using SLURM makes the training run independent of the local computer state.

This means:
- the job keeps running even if the laptop sleeps
- the job is not interrupted by SSH disconnection
- resource requests are explicit and reproducible

**Strategy**

I create a small SLURM submission script that:
- requests one GPU
- targets a GPU type compatible with the current BEVFusion build
- sets an appropriate wall-time limit
- activates the correct conda environment
- sets `PYTHONPATH` for the BEVFusion project import
- launches the MMDetection3D training command
- can resume from the latest checkpoint if needed

This is important because BEVFusion training is computationally heavier than the LiDAR-only baseline and may require multiple sessions depending on the available GPU and wall-time limit.

## Defining the SLURM script location

I store the SLURM submission script inside the project repository so that the full training procedure remains reproducible and well documented.

This ensures that:
- the exact execution setup is version-controlled together with the code
- the same script can be reused across different experiments
- running conditions (resources, environment, command) are explicitly defined and easy to reproduce

Keeping the SLURM script within the project structure also simplifies debugging and future extensions of the training pipeline.

In [15]:
WORK_DIR: Final[Path] = MMDET3D_ROOT / "work_dirs" / "bevfusion_nuscenes_20pct"
TRAIN_CONFIG_PATH: Final[Path] = MMDET3D_ROOT / "configs" / "my_experiments" / "bevfusion_nuscenes_20pct.py"

SLURM_DIR: Final[Path] = PROJECT_ROOT / "slurm"
SLURM_RUN_DIR: Final[Path] = SLURM_DIR / "bevfusion_20pct"
SLURM_RUN_DIR.mkdir(parents=True, exist_ok=True)

SLURM_SCRIPT_PATH: Final[Path] = SLURM_RUN_DIR / "train_bevfusion_nuscenes_20pct.slurm"

print("WORK_DIR         :", WORK_DIR)
print("TRAIN_CONFIG_PATH:", TRAIN_CONFIG_PATH)
print("SLURM_RUN_DIR    :", SLURM_RUN_DIR)
print("SLURM_SCRIPT_PATH:", SLURM_SCRIPT_PATH)

WORK_DIR         : /storage/homefs/ae04q066/projects/nuscenes-multimodal-learning/external/mmdetection3d/work_dirs/bevfusion_nuscenes_20pct
TRAIN_CONFIG_PATH: /storage/homefs/ae04q066/projects/nuscenes-multimodal-learning/external/mmdetection3d/configs/my_experiments/bevfusion_nuscenes_20pct.py
SLURM_RUN_DIR    : /storage/homefs/ae04q066/projects/nuscenes-multimodal-learning/slurm/bevfusion_20pct
SLURM_SCRIPT_PATH: /storage/homefs/ae04q066/projects/nuscenes-multimodal-learning/slurm/bevfusion_20pct/train_bevfusion_nuscenes_20pct.slurm


## Creating a resume-ready SLURM submission script

The script below is designed for a single-GPU UBELIX training run with a 12-hour wall-time limit.

UBELIX requires the GPU type to be specified explicitly in the SLURM request.
For this experiment, I request one A100 GPU, which is compatible with the current BEVFusion setup.

The script supports both:
- a fresh training run if no checkpoint is present
- a resumed training run if a previous checkpoint already exists

https://slurm.schedmd.com/sbatch.html#OPT_signal 

https://hpc-unibe-ch.github.io/runjobs/scheduled-jobs/preemption/ 

In [22]:
SLURM_SCRIPT: str = f"""#!/bin/bash
#
# Description:
#   SLURM submission script for BEVFusion training on nuScenes (20% subset).
#   Runs on a single A100 GPU with automatic checkpoint resume and
#   preemption-safe requeue behavior.
#
# Usage:
#   sbatch {SLURM_SCRIPT_PATH}
#
# Check job status:
#   squeue -u $USER
#
# Follow logs:
#   tail -f {SLURM_RUN_DIR}/job_<JOBID>.out
#   tail -f {SLURM_RUN_DIR}/job_<JOBID>.err
#
# Check whether a checkpoint exists:
#   ls -lh {WORK_DIR}
#   cat {WORK_DIR}/last_checkpoint
#
# Resume behavior test:
#   1. Submit the job with:
#        sbatch {SLURM_SCRIPT_PATH}
#   2. Wait until a checkpoint is written.
#   3. Cancel the job manually with:
#        scancel <JOBID>
#   4. Submit the same script again:
#        sbatch {SLURM_SCRIPT_PATH}
#   5. Confirm that the log contains:
#        Resuming training from ...
#
#SBATCH --job-name=bevfusion_20pct
#SBATCH --output={SLURM_RUN_DIR}/job_%j.out
#SBATCH --error={SLURM_RUN_DIR}/job_%j.err
#SBATCH --time=12:00:00
#SBATCH --nodes=1
#SBATCH --gpus-per-node=a100:1
#SBATCH --cpus-per-task=16
#SBATCH --mem=48G
#SBATCH --qos=job_gpu_preemptable
#SBATCH --signal=SIGUSR1@120
#SBATCH --requeue

source ~/.bashrc
conda activate py38_mmdet3d

cd {MMDET3D_ROOT}
export PYTHONPATH=$PWD:$PYTHONPATH
echo "Working directory: $(pwd)"

LAST_CHECKPOINT_FILE="{WORK_DIR}/last_checkpoint"

handle_signal() {{
    echo "Received preemption signal. Exiting so Slurm can requeue."
    exit 0
}}

trap handle_signal SIGUSR1

if [ -f "$LAST_CHECKPOINT_FILE" ]; then
    CHECKPOINT_PATH=$(cat "$LAST_CHECKPOINT_FILE")
    echo "Resuming training from $CHECKPOINT_PATH"
    python tools/train.py {TRAIN_CONFIG_PATH} \\
        --resume "$CHECKPOINT_PATH"
else
    echo "No checkpoint found. Starting fresh training run."
    python tools/train.py {TRAIN_CONFIG_PATH}
fi
"""

In [23]:
print("TRAIN_CONFIG_PATH exists :", TRAIN_CONFIG_PATH.exists())
print("WORK_DIR exists         :", WORK_DIR.exists())

last_ckpt_path = WORK_DIR / "last_checkpoint"
epoch_ckpt_path = WORK_DIR / "epoch_5.pth"

print("last_checkpoint exists  :", last_ckpt_path.exists())
print("epoch_5 exists          :", epoch_ckpt_path.exists())

if last_ckpt_path.exists():
    print("last_checkpoint content:")
    print(last_ckpt_path.read_text().strip())

TRAIN_CONFIG_PATH exists : True
WORK_DIR exists         : True
last_checkpoint exists  : False
epoch_5 exists          : False


In [24]:
with open(SLURM_SCRIPT_PATH, "w") as f:
    f.write(SLURM_SCRIPT)

print("Saved SLURM script to:")
print(SLURM_SCRIPT_PATH)

Saved SLURM script to:
/storage/homefs/ae04q066/projects/nuscenes-multimodal-learning/slurm/bevfusion_20pct/train_bevfusion_nuscenes_20pct.slurm


## Verifying the generated SLURM script

Before submitting the job, I inspect the generated script to confirm that:
- the requested resources match the current BEVFusion training setup
- the requested GPU type is compatible with the current build
- the environment activation is correct
- `PYTHONPATH` is set correctly for the BEVFusion project import
- the training command points to the correct experiment config
- the script checks for an existing checkpoint and resumes automatically when available

In [25]:
with open(SLURM_SCRIPT_PATH, "r") as f:
    slurm_lines = f.readlines()

for i, line in enumerate(slurm_lines):
    print(f"{i+1:02d}: {line.rstrip()}")

01: #!/bin/bash
02: #
03: # Description:
04: #   SLURM submission script for BEVFusion training on nuScenes (20% subset).
05: #   Runs on a single A100 GPU with automatic checkpoint resume and
06: #   preemption-safe requeue behavior.
07: #
08: # Usage:
09: #   sbatch /storage/homefs/ae04q066/projects/nuscenes-multimodal-learning/slurm/bevfusion_20pct/train_bevfusion_nuscenes_20pct.slurm
10: #
11: # Check job status:
12: #   squeue -u $USER
13: #
14: # Follow logs:
15: #   tail -f /storage/homefs/ae04q066/projects/nuscenes-multimodal-learning/slurm/bevfusion_20pct/job_<JOBID>.out
16: #   tail -f /storage/homefs/ae04q066/projects/nuscenes-multimodal-learning/slurm/bevfusion_20pct/job_<JOBID>.err
17: #
18: # Check whether a checkpoint exists:
19: #   ls -lh /storage/homefs/ae04q066/projects/nuscenes-multimodal-learning/external/mmdetection3d/work_dirs/bevfusion_nuscenes_20pct
20: #   cat /storage/homefs/ae04q066/projects/nuscenes-multimodal-learning/external/mmdetection3d/work_dirs/bevfu

## SLURM submission and resume behavior

The training job can be submitted by navigating to the project directory and launching the SLURM script:

`cd /storage/homefs/ae04q066/projects/nuscenes-multimodal-learning`  
`sbatch slurm/lidar_20pct/train_lidar_baseline_nuscenes_20pct.slurm`

After submission, the job can be monitored using standard SLURM commands such as `squeue -u ae04q066`.

Training progress can be followed through the output log using:  
`tail -f /storage/homefs/ae04q066/projects/nuscenes-multimodal-learning/slurm/lidar_20pct/job_<JOBID>.out`

Errors can be inspected with:  
`cat /storage/homefs/ae04q066/projects/nuscenes-multimodal-learning/slurm/lidar_20pct/job_<JOBID>.err`

The SLURM script automatically checks whether the following file exists:

`/storage/homefs/ae04q066/projects/nuscenes-multimodal-learning/external/mmdetection3d/work_dirs/lidar_baseline_nuscenes_20pct/last_checkpoint`

This file contains the path to the most recent checkpoint saved during training. If `last_checkpoint` exists, the script reads the stored path and resumes training from that checkpoint. If it does not exist, training starts from scratch. This mechanism allows training to continue seamlessly across multiple 12-hour SLURM jobs without manually modifying the training command.

A running job can be stopped using `scancel <JOBID>`.

This execution setup is robust to laptop sleep, SSH disconnections, unstable network connections, and wall-time limits requiring multiple job submissions.